In [0]:
# ==========================================
# 01_Raw_to_Bronze.py
# Member 1: Ingestion & Bronze Layer
# ==========================================

from pyspark.sql.types import (
    StructType, StructField, StringType, 
    DoubleType, IntegerType, ArrayType
)
from pyspark.sql.functions import current_timestamp, lit

# 1. Define Strict Schema (Schema-on-Read)
# This perfectly matches the 5-city JSON payload.

# Define the structure for the "hourly_units" object
hourly_units_schema = StructType([
    StructField("time", StringType(), True),
    StructField("carbon_monoxide", StringType(), True),
    StructField("carbon_dioxide", StringType(), True),
    StructField("pm2_5", StringType(), True),
    StructField("pm10", StringType(), True),
    StructField("nitrogen_dioxide", StringType(), True)
])

# Define the structure for the "hourly" object (arrays of data)
hourly_schema = StructType([
    StructField("time", ArrayType(StringType()), True),
    StructField("carbon_monoxide", ArrayType(DoubleType()), True),
    StructField("carbon_dioxide", ArrayType(DoubleType()), True), # Casted to Double for safety, even if API sends Ints
    StructField("pm2_5", ArrayType(DoubleType()), True),
    StructField("pm10", ArrayType(DoubleType()), True),
    StructField("nitrogen_dioxide", ArrayType(DoubleType()), True)
])

# Define the main root structure of the JSON
open_meteo_schema = StructType([
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("generationtime_ms", DoubleType(), True),
    StructField("utc_offset_seconds", IntegerType(), True),
    StructField("timezone", StringType(), True),
    StructField("timezone_abbreviation", StringType(), True),
    StructField("elevation", DoubleType(), True),
    StructField("hourly_units", hourly_units_schema, True),
    StructField("hourly", hourly_schema, True),
    StructField("city", StringType(), True),
    StructField("source_system", StringType(), True)
])

print("Strict Schema defined successfully and matches the verified JSON payload!")

Strict Schema defined successfully and matches the verified JSON payload!


In [0]:
# ==========================================
# Sanity Check: Test the Schema against the file
# ==========================================

# Replace this string with the actual path to your JSON file in Databricks
file_path = "/Volumes/workspace/default/bronze/incremental_load_sample.json"

try:
    # Read the JSON file using our strict schema and the multiline option 
    # (multiline=True is required because the JSON array spans multiple lines)
    test_df = (spark.read
               .schema(open_meteo_schema)
               .option("multiline", "true")
               .json(file_path))
    
    print("File read successfully! Here is a preview:")
    display(test_df)
    
except Exception as e:
    print("Error reading the file. Check your file path!")
    print(e)

File read successfully! Here is a preview:


latitude,longitude,generationtime_ms,utc_offset_seconds,timezone,timezone_abbreviation,elevation,hourly_units,hourly,city,source_system
31.5,74.3,0.22983551025390625,18000,Asia/Karachi,GMT+5,222.0,"List(iso8601, μg/m³, ppm, μg/m³, μg/m³, μg/m³)","List(List(2026-09-24T00:00, 2026-09-24T01:00, 2026-09-24T02:00, 2026-09-24T03:00, 2026-09-24T04:00, 2026-09-24T05:00, 2026-09-24T06:00, 2026-09-24T07:00, 2026-09-24T08:00, 2026-09-24T09:00, 2026-09-24T10:00, 2026-09-24T11:00, 2026-09-24T12:00, 2026-09-24T13:00, 2026-09-24T14:00, 2026-09-24T15:00, 2026-09-24T16:00, 2026-09-24T17:00, 2026-09-24T18:00, 2026-09-24T19:00, 2026-09-24T20:00, 2026-09-24T21:00, 2026-09-24T22:00, 2026-09-24T23:00), List(3764.0, 3512.0, 3120.0, 2512.0, 1765.0, 1229.0, 1108.0, 1197.0, 1236.0, 1103.0, 920.0, 790.0, 749.0, 762.0, 827.0, 951.0, 1126.0, 1330.0, 1612.0, 1922.0, 2091.0, 1987.0, 1741.0, 1584.0), List(515.0, 514.0, 511.0, 505.0, 496.0, 487.0, 480.0, 473.0, 467.0, 462.0, 457.0, 454.0, 453.0, 455.0, 456.0, 457.0, 459.0, 461.0, 465.0, 471.0, 474.0, 473.0, 471.0, 471.0), List(112.2, 119.2, 122.0, 123.9, 123.9, 80.1, 85.9, 86.4, 74.7, 66.4, 55.3, 49.1, 46.8, 44.6, 42.4, 40.2, 39.3, 41.2, 47.8, 52.0, 50.8, 49.4, 50.9, 54.0), List(123.9, 130.9, 133.6, 135.5, 135.5, 89.2, 94.3, 93.4, 84.5, 78.2, 67.8, 60.8, 58.2, 55.6, 52.9, 50.1, 48.7, 49.1, 53.7, 56.8, 54.9, 53.4, 55.1, 58.4), List(99.3, 104.8, 107.0, 105.0, 99.5, 48.9, 47.0, 44.0, 38.5, 28.2, 15.5, 6.3, 3.2, 3.6, 5.8, 9.2, 14.3, 21.2, 32.0, 44.7, 52.9, 52.6, 48.0, 45.0))",Lahore,open-meteo-air-quality-api
28.599998,77.20001,0.17595291137695312,19800,Asia/Kolkata,GMT+5:30,214.0,"List(iso8601, μg/m³, ppm, μg/m³, μg/m³, μg/m³)","List(List(2026-09-24T00:00, 2026-09-24T01:00, 2026-09-24T02:00, 2026-09-24T03:00, 2026-09-24T04:00, 2026-09-24T05:00, 2026-09-24T06:00, 2026-09-24T07:00, 2026-09-24T08:00, 2026-09-24T09:00, 2026-09-24T10:00, 2026-09-24T11:00, 2026-09-24T12:00, 2026-09-24T13:00, 2026-09-24T14:00, 2026-09-24T15:00, 2026-09-24T16:00, 2026-09-24T17:00, 2026-09-24T18:00, 2026-09-24T19:00, 2026-09-24T20:00, 2026-09-24T21:00, 2026-09-24T22:00, 2026-09-24T23:00), List(1065.0, 900.0, 759.0, 678.0, 620.0, 593.0, 626.0, 691.0, 711.0, 635.0, 514.0, 423.0, 393.0, 393.0, 410.0, 447.0, 501.0, 546.0, 576.0, 597.0, 605.0, 598.0, 579.0, 551.0), List(472.0, 470.0, 469.0, 471.0, 474.0, 475.0, 471.0, 464.0, 458.0, 453.0, 448.0, 445.0, 445.0, 446.0, 448.0, 450.0, 452.0, 453.0, 451.0, 447.0, 445.0, 445.0, 447.0, 449.0), List(102.7, 91.3, 76.1, 66.4, 58.3, 65.9, 63.6, 61.7, 50.8, 39.5, 37.1, 37.6, 38.2, 37.8, 39.6, 38.5, 38.4, 40.7, 45.1, 45.1, 42.6, 40.2, 38.8, 36.6), List(122.0, 116.9, 112.7, 110.4, 102.4, 91.3, 83.6, 77.5, 63.8, 48.8, 45.6, 46.8, 48.0, 51.8, 51.7, 51.7, 49.1, 48.4, 52.7, 54.6, 56.7, 60.8, 64.3, 65.2), List(53.1, 41.4, 31.9, 27.5, 25.2, 34.6, 33.9, 32.8, 29.9, 23.8, 16.0, 10.1, 7.3, 6.4, 7.6, 11.6, 17.6, 23.3, 28.5, 33.5, 36.5, 36.6, 34.7, 32.0))",Delhi,open-meteo-air-quality-api
39.90001,116.399994,0.33223628997802734,28800,Asia/Shanghai,GMT+8,47.0,"List(iso8601, μg/m³, ppm, μg/m³, μg/m³, μg/m³)","List(List(2026-09-24T00:00, 2026-09-24T01:00, 2026-09-24T02:00, 2026-09-24T03:00, 2026-09-24T04:00, 2026-09-24T05:00, 2026-09-24T06:00, 2026-09-24T07:00, 2026-09-24T08:00, 2026-09-24T09:00, 2026-09-24T10:00, 2026-09-24T11:00, 2026-09-24T12:00, 2026-09-24T13:00, 2026-09-24T14:00, 2026-09-24T15:00, 2026-09-24T16:00, 2026-09-24T17:00, 2026-09-24T18:00, 2026-09-24T19:00, 2026-09-24T20:00, 2026-09-24T21:00, 2026-09-24T22:00, 2026-09-24T23:00), List(1162.0, 1217.0, 1232.0, 1192.0, 1113.0, 1003.0, 838.0, 642.0, 504.0, 473.0, 500.0, 519.0, 496.0, 466.0, 465.0, 496.0, 558.0, 680.0, 919.0, 1219.0, 1442.0, 1539.0, 1559.0, 1516.0), List(505.0, 510.0, 512.0, 509.0, 503.0, 496.0, 487.0, 477.0, 469.0, 466.0, 466.0, 466.0, 465.0, 463.0, 463.0, 464.0, 466.0, 471.0, 481.0, 493.0, 503.0, 507.0, 509.0, 508.0), List(154.1, 156.9, 156.9, 142.0, 130.9, 121.5, 103.8, 102.2, 93.6, 79.2, 53.4, 48.2, 40.5, 38.9, 39.1, 40.4, 43.8, 48.1, 68.7,

In [0]:
# ==========================================
# Create Audit Log Table (Run once)
# ==========================================
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType

log_table_path = "/Volumes/YOUR_CATALOG/YOUR_SCHEMA/YOUR_VOLUME/pipeline_execution_logs"

# Define the schema based exactly on the Phase 2 requirements
log_schema = StructType([
    StructField("layer_processed", StringType(), True),
    StructField("parameter_processed", StringType(), True),
    StructField("start_time", TimestampType(), True),
    StructField("end_time", TimestampType(), True),
    StructField("status", StringType(), True),
    StructField("rows_processed", IntegerType(), True)
])

# Create an empty DataFrame and save it as a Delta table (if it doesn't exist)
try:
    spark.read.format("delta").load(log_table_path)
    print("Log table already exists.")
except:
    empty_df = spark.createDataFrame([], log_schema)
    empty_df.write.format("delta").save(log_table_path)
    print("Audit Log table created successfully!")

# ==========================================
# FINAL Parameterized Bronze Ingestion Function (With Logging)
# ==========================================
from pyspark.sql.functions import current_timestamp, lit
from datetime import datetime
from pyspark.sql import functions as F

def ingest_raw_to_bronze(input_file_path, bronze_table_path, log_table_path):
    layer_name = "Raw-to-Bronze"
    start_time = datetime.now()
    status = "Failed"
    row_count = 0
    
    print(f"--- Starting {layer_name} ---")
    
    try:
        # 1. Read & Add Metadata
        raw_df = (spark.read
                  .schema(open_meteo_schema)
                  .option("multiline", "true")
                  .json(input_file_path))
        
        bronze_df = (raw_df
                     .withColumn("load_timestamp", current_timestamp())
                     .withColumn("source_file", F.col("_metadata.file_path")))
        
        # 2. Write to Bronze
        (bronze_df.write
         .format("delta")
         .mode("append") 
         .option("mergeSchema", "true") 
         .save(bronze_table_path))
        
        row_count = bronze_df.count()
        status = "Success"
        print(f"Success! {row_count} rows appended to Bronze Layer.")
        
    except Exception as e:
        print(f"Pipeline Failed: {e}")
        status = f"Failed: {str(e)[:50]}" # Log first 50 chars of error
        
    finally:
        end_time = datetime.now()
        
        # 3. Write to Audit Log (Fulfills Phase 2 Audit Requirement)
        log_entry = spark.createDataFrame([(
            layer_name, 
            input_file_path, 
            start_time, 
            end_time, 
            status, 
            row_count
        )], schema=log_schema)
        
        log_entry.write.format("delta").mode("append").save(log_table_path)
        print("Audit Log updated.")
        print(f"---------------------------------")
        
    return status

Log table already exists.


In [0]:
# ==========================================
# 3. Execute the Pipeline (Simulating a run)
# ==========================================

# 1. Define your parameters
# (Change these paths to your actual Unity Catalog Volume paths)
source_data_parameter = "/Volumes/workspace/default/bronze/incremental_load_sample.json"
bronze_delta_path = "/Volumes/workspace/default/bronze_layer/bronze_carbon_data"
log_table_path = "/Volumes/workspace/default/bronze_layer/pipeline_execution_logs"

ingest_raw_to_bronze(source_data_parameter, bronze_delta_path, log_table_path)

# Verify the log was written!
display(spark.read.format("delta").load(log_table_path))

--- Starting Raw-to-Bronze ---
Success! 5 rows appended to Bronze Layer.
Audit Log updated.
---------------------------------


layer_processed,parameter_processed,start_time,end_time,status,rows_processed
Raw-to-Bronze,/Volumes/workspace/default/bronze/incremental_load_sample.json,2026-10-01T18:57:28.216Z,2026-10-01T18:57:31.734Z,Success,5
